# Lot D · Silver : fiche par LLM (Qwen2.5-7B-Instruct, 4 bits)

Colab : Exécution → Modifier le type d'exécution → **GPU T4**. Dev uniquement : le test gelé n'est jamais lu.

**Étape 2** : schéma « champs » (le LLM extrait des valeurs recopiées, le code compose les phrases de la fiche).
Une seule génération sur GPU (A) ; la variante B (`--completer`) rejoue les mêmes réponses sans GPU.

In [ ]:
!nvidia-smi -L

## 1. Récupérer la branche `lot-d-silver` (cellule ré-exécutable)

In [ ]:
%cd /content
import os
if os.path.isdir('fardi-caseforge/.git'):
    !git -C fardi-caseforge pull -q origin lot-d-silver
else:
    !git clone -q -b lot-d-silver https://github.com/onsalouini/fardi-caseforge.git
%cd /content/fardi-caseforge
!git log --oneline -1

## 2. Dépendances (torch de Colab conservé)

In [ ]:
!pip install -q "transformers>=4.44" accelerate bitsandbytes

## 3. Essai rapide sur 2 documents du dev
Chaque ligne doit afficher `essais 1` et `repli False`.

In [ ]:
!python generation/llm.py --split dev --graine 42 --limite 2 --out generation/sorties/essai.jsonl

## 4. A · dev complet, schéma champs, graine 42 (exemples=3, entités, contrôle d'ancrage)

In [ ]:
!python generation/llm.py --split dev --schema champs --graine 42 --out generation/sorties/llm_champs_dev_s42.jsonl

## 5. B · mêmes réponses + complétion (rejeu sans GPU)

In [ ]:
!python generation/llm.py --split dev --schema champs --graine 42 --completer --rejouer generation/sorties/llm_champs_dev_s42.journal.jsonl --out generation/sorties/llm_champs_compl_dev_s42.jsonl

## 6. Évaluation : gabarit v2, A, B

In [ ]:
!python generation/gabarit.py --split dev --out generation/sorties/gabarit_v2_dev.jsonl
for nom in ['gabarit_v2_dev', 'llm_champs_dev_s42', 'llm_champs_compl_dev_s42']:
    print('=' * 20, nom)
    !python generation/evaluer.py --gen generation/sorties/{nom}.jsonl --out generation/sorties/eval_{nom}.md | sed -n '6,17p'

## 7. Résumé du journal (A)

In [ ]:
import json
j = [json.loads(l) for l in open('generation/sorties/llm_champs_dev_s42.journal.jsonl', encoding='utf-8')]
print('replis gabarit       :', sum(x['repli_gabarit'] for x in j))
print('essais > 1           :', sum(x['essais'] > 1 for x in j))
print('retirés (non ancrés) :', sum(len(x.get('retires_non_ancres', [])) for x in j))
print('citations corrigées  :', sum(len(x.get('citations_corrigees', [])) for x in j))
print('retirés (sensibles)  :', sum(len(x.get('retires_sensibles', [])) for x in j))
print('temps moyen / doc (s):', round(sum(x['temps_s'] for x in j) / len(j), 1))

## 8. Télécharger les sorties
Dézipper dans `generation/` du dépôt local (le dossier `sorties/` est ignoré par Git) : les journaux contiennent les réponses du LLM, qui servent à l'analyse d'erreurs et aux ablations sans GPU.

In [ ]:
!cd generation && zip -qr sorties_etape2.zip sorties
from google.colab import files
files.download('generation/sorties_etape2.zip')